In [50]:
import pandas as pd

df_ground_truth = pd.read_csv("../data/ground_truth.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [51]:
ground_truth[10]

{'question': 'Where do I watch the Office Hours or live workshop stream if I’m a student?',
 'document': '489dd1c9d9'}

In [52]:
from ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

In [53]:
doc_idx = {}

for doc in documents:
    doc_idx[doc["id"]] = doc

In [54]:
q = ground_truth[10]
q

{'question': 'Where do I watch the Office Hours or live workshop stream if I’m a student?',
 'document': '489dd1c9d9'}

In [55]:
doc_idx[q['document']]

{'id': '489dd1c9d9',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'What is the video/zoom link to the stream for the “Office Hours” or live/workshop sessions?',
 'answer': 'The zoom link is only published to instructors/presenters/TAs.\n\nStudents participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the [announcements channel on Telegram and Slack](https://t.me/dezoomcamp) before it begins. You can also watch live on the DataTalksClub [YouTube Channel](https://www.youtube.com/c/DataTalksClub).\n\nDon’t post questions in chat as they may be missed if the room is very active.'}

In [56]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [57]:
from evaluation_utils import RAGWithUsage

assistant = RAGWithUsage(
    index=index,
    llm_client=openai_client,
    course='llm-zoomcamp',
)

In [58]:
q['question']

'Where do I watch the Office Hours or live workshop stream if I’m a student?'

In [59]:
answer = assistant.rag(q['question'])

In [60]:
assistant.total_cost()

0.000846

In [61]:
print(answer)

Students participate via YouTube Live. The video URL is posted in the announcements channel on Telegram and Slack before it begins, and you can also watch live on the DataTalksClub YouTube Channel.


In [62]:
doc_id = q["document"]
original_doc = doc_idx[doc_id]
answer_orig = original_doc["answer"]

answer_orig

'The zoom link is only published to instructors/presenters/TAs.\n\nStudents participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the [announcements channel on Telegram and Slack](https://t.me/dezoomcamp) before it begins. You can also watch live on the DataTalksClub [YouTube Channel](https://www.youtube.com/c/DataTalksClub).\n\nDon’t post questions in chat as they may be missed if the room is very active.'

In [63]:
rag_result = {
    "question": q['question'],
    "answer_llm": answer,
    "answer_orig": answer_orig,
    "document": doc_id,
}

rag_result

{'question': 'Where do I watch the Office Hours or live workshop stream if I’m a student?',
 'answer_llm': 'Students participate via YouTube Live. The video URL is posted in the announcements channel on Telegram and Slack before it begins, and you can also watch live on the DataTalksClub YouTube Channel.',
 'answer_orig': 'The zoom link is only published to instructors/presenters/TAs.\n\nStudents participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the [announcements channel on Telegram and Slack](https://t.me/dezoomcamp) before it begins. You can also watch live on the DataTalksClub [YouTube Channel](https://www.youtube.com/c/DataTalksClub).\n\nDon’t post questions in chat as they may be missed if the room is very active.',
 'document': '489dd1c9d9'}

In [64]:
def generate_rag_answer(rec):
    question = rec["question"]
    doc_id = rec["document"]
    original_doc = doc_idx[doc_id]

    answer_llm = assistant.rag(question)
    answer_orig = original_doc["answer"]

    result = {
        "question": question,
        "answer_llm": answer_llm,
        "answer_orig": answer_orig,
        "document": doc_id,
    }

    return result

In [65]:
record = generate_rag_answer(q)
record

{'question': 'Where do I watch the Office Hours or live workshop stream if I’m a student?',
 'answer_llm': 'Students watch the Office Hours or live workshop stream on **YouTube Live**. The video URL is posted in the **announcements channel on Telegram and Slack** before it starts, and you can also watch it on the **DataTalksClub YouTube channel**.\n\nThe **Zoom link is only for instructors/presenters/TAs**.',
 'answer_orig': 'The zoom link is only published to instructors/presenters/TAs.\n\nStudents participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the [announcements channel on Telegram and Slack](https://t.me/dezoomcamp) before it begins. You can also watch live on the DataTalksClub [YouTube Channel](https://www.youtube.com/c/DataTalksClub).\n\nDon’t post questions in chat as they may be missed if the room is very active.',
 'document': '489dd1c9d9'}

In [66]:
q2 = ground_truth[31]

In [67]:
q2

{'question': 'Do I have to do all the homework assignments to receive the certificate?',
 'document': '9f689c185f'}

In [68]:

experiment = generate_rag_answer(q2)

In [69]:
answer2 = assistant.rag(q2['question'])

In [70]:
print(answer2)

No. You only need to pass the Capstone project to receive the certificate. Homework is not mandatory, though it is recommended.


In [71]:
doc_id2 = q2["document"]
original_doc = doc_idx[doc_id2]
answer_orig2 = original_doc["answer"]

answer_orig2

'Yes, you need to pass the Capstone project to get the certificate. Homework is not mandatory, though it is recommended for reinforcing concepts, and the points awarded count towards your rank on the leaderboard.'

In [72]:
rag_result2 = {
    "question": q2['question'],
    "answer_llm": answer2,
    "answer_orig": answer_orig2,
    "document": doc_id2,
}

rag_result2

{'question': 'Do I have to do all the homework assignments to receive the certificate?',
 'answer_llm': 'No. You only need to pass the Capstone project to receive the certificate. Homework is not mandatory, though it is recommended.',
 'answer_orig': 'Yes, you need to pass the Capstone project to get the certificate. Homework is not mandatory, though it is recommended for reinforcing concepts, and the points awarded count towards your rank on the leaderboard.',
 'document': '9f689c185f'}

In [73]:
assistant.total_cost()

0.0028935

In [74]:
assistant.reset_usage()

In [75]:
assistant.total_cost()

0.0

In [76]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [35]:
ground_truth[31]

{'question': 'Do I have to do all the homework to receive the certificate for this course?',
 'document': '9f689c185f'}

In [77]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, ground_truth, generate_rag_answer)

  0%|          | 0/565 [00:00<?, ?it/s]

In [78]:
results[:10]

[{'question': 'Can I still join the course if I just found out about it?',
  'answer_llm': 'Yes, you can still join the course. If you want a certificate, you need to submit your project while submissions are still being accepted.',
  'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
  'document': '74eb249bbf'},
 {'question': 'If I join late, am I still eligible for a certificate?',
  'answer_llm': 'No. You can join late, but you can only get a certificate if you finish the course with a live cohort and submit your project while submissions are still open.',
  'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
  'document': '74eb249bbf'},
 {'question': 'Do I need to submit my project before submissions close to get the certificate?',
  'answer_llm': 'Yes. To get the certificate, you need to submit your project whi

In [79]:
df_results = pd.DataFrame(results)

In [80]:
df_results.head()

,question,answer_llm,answer_orig,document
0,Can I still join the course if I just found ou...,"Yes, you can still join the course. If you wan...","Yes, but if you want to receive a certificate,...",74eb249bbf
1,"If I join late, am I still eligible for a cert...","No. You can join late, but you can only get a ...","Yes, but if you want to receive a certificate,...",74eb249bbf
2,Do I need to submit my project before submissi...,"Yes. To get the certificate, you need to submi...","Yes, but if you want to receive a certificate,...",74eb249bbf
3,Is it okay to start the course now even though...,"Yes — you can start whenever you want, even if...","Yes, but if you want to receive a certificate,...",74eb249bbf
4,What’s the rule for getting a certificate if I...,"If you join after the course has started, you ...","Yes, but if you want to receive a certificate,...",74eb249bbf


In [81]:
assistant.total_cost()


0.6096960000000005

In [83]:
df_results.to_csv("../data/rag-answers-mine.csv", index=False)